In [1]:
# Install required packages
!pip install -q langchain langchain-openai langgraph langchain-tavily pydantic

In [2]:
# Import the libraries needed for ModelRank

import os
import json
import time
import requests

from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import TypedDict, Literal

from pydantic import BaseModel, Field, ValidationError

from google.colab import userdata
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, END

In [3]:
# Load API keys from Colab Secrets

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

In [4]:
# Define the language model used by the agents

model = ChatOpenAI(
    model="google/gemini-3.8-flash",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

In [5]:

# Structured Output - Schemas

from pydantic import BaseModel, Field, ConfigDict
from typing import Optional


# Shared model information

class CandidateModel(BaseModel):
    model_config = ConfigDict(extra="forbid")

    name: str
    provider: str
    model_id: str


# 1. Planner Agent

class EvaluationPlan(BaseModel):
    project_name: str
    project_type: str
    project_goal: str
    evaluation_criteria: list[str]
    candidate_models: list[CandidateModel]
    notes: str = ""


# 2. Benchmark Agent

class BenchmarkCase(BaseModel):
    model_config = ConfigDict(extra="forbid")

    id: str
    criterion: str
    prompt: str
    expected_behavior: str


class BenchmarkDocument(BaseModel):
    project_name: str
    test_cases: list[BenchmarkCase] = Field(
        min_length=3,
        max_length=3
    )


# 3. Runner Agent

class TestExecutionResult(BaseModel):
    test_case_id: str
    criterion: str
    expected_behavior: str
    output: Optional[str] = None
    latency_seconds: float
    input_tokens: Optional[int] = None
    output_tokens: Optional[int] = None
    total_tokens: Optional[int] = None
    cost_usd: Optional[float] = None
    error: Optional[str] = None


class RunnerSummary(BaseModel):
    average_latency_seconds: Optional[float] = None
    total_cost_usd: Optional[float] = None
    total_input_tokens: int
    total_output_tokens: int
    successful_tests: int
    failed_tests: int


class ModelExecutionResults(CandidateModel):
    results: list[TestExecutionResult]
    summary: Optional[RunnerSummary] = None


class RunnerResults(BaseModel):
    project_name: str
    models: list[ModelExecutionResults]


# 4. Judge Agent

class ModelRanking(CandidateModel):
    rank: int = Field(ge=1)
    quality_assessment: str
    average_latency_seconds: Optional[float] = None
    total_cost_usd: Optional[float] = None
    reason: str


class FinalDecision(BaseModel):
    selected_model: Optional[CandidateModel]
    model_rankings: list[ModelRanking]
    decision_reason: str
    retry_required: bool
    retry_reason: str = ""


# Agent 1 - Planner Agent

In [6]:

# Planner Agent - Tools

tavily_search = TavilySearch(max_results=5)


@tool
def search_models(query: str) -> str:
    """Search the web for current information about suitable LLMs."""

    results = tavily_search.invoke(
        {
            "query": query
        }
    )

    return json.dumps(
        results,
        ensure_ascii=False
    )


@tool
def verify_model_ids(candidate_models: list[dict]) -> str:
    """Verify that candidate model IDs currently exist on OpenRouter."""

    response = requests.get(
        "https://openrouter.ai/api/v1/models",
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    available_model_ids = {
        model_info["id"]
        for model_info in data.get("data", [])
        if model_info.get("id")
    }

    verified_models = []
    invalid_models = []

    for candidate in candidate_models:
        model_id = candidate.get("model_id")

        if model_id in available_model_ids:
            verified_models.append(candidate)
        else:
            invalid_models.append(candidate)

    result = {
        "verified_models": verified_models,
        "invalid_models": invalid_models
    }

    return json.dumps(
        result,
        ensure_ascii=False
    )


@tool(args_schema=EvaluationPlan)
def save_evaluation_plan(
    project_name: str,
    project_type: str,
    project_goal: str,
    evaluation_criteria: list[str],
    candidate_models: list[dict],
    notes: str = ""
) -> str:
    """Validate and save the evaluation plan as a JSON file."""

    plan = EvaluationPlan.model_validate(
        {
            "project_name": project_name,
            "project_type": project_type,
            "project_goal": project_goal,
            "evaluation_criteria": evaluation_criteria,
            "candidate_models": candidate_models,
            "notes": notes
        }
    )

    with open("evaluation_plan.json", "w", encoding="utf-8") as file:
        json.dump(
            plan.model_dump(),
            file,
            indent=2,
            ensure_ascii=False
        )

    return "Evaluation plan saved to evaluation_plan.json"


In [7]:

# Planner Agent - Prompt

PLANNER_PROMPT = """
You are the Planner Agent in ModelRank.

Your task is to analyze the user's AI project and create
a relevant evaluation plan.

Identify the project's goal, type, evaluation criteria,
and suitable candidate LLMs.

Use only requirements explicitly provided by the user.
Do not invent requirements, constraints, examples, or assumptions.
Evaluation criteria must come from the user's request,
not from external search results.

Use search_models to find suitable LLMs and their exact
OpenRouter model IDs.

Search only once unless the results are insufficient
or model verification fails.

Select distinct, relevant models supported by the search results.
Prefer different providers when appropriate.

Never invent model names or IDs.
Each model's name, provider, and model_id must match.
Use exact OpenRouter IDs without the "openrouter:" prefix.

Always call verify_model_ids before saving the plan.

Include only models returned in verified_models.
If any model is invalid, search for a replacement
and verify again.

Never save unverified models.

Do not claim that candidate models have been tested,
benchmarked, ranked, or validated by ModelRank.

Keep selection notes brief and based on search evidence.

Focus only on planning.
Do not generate test cases or execute evaluations.

Once verification succeeds, call save_evaluation_plan
to save evaluation_plan.json.

After saving, respond with exactly:
"Evaluation plan saved to evaluation_plan.json"

Do not summarize, explain, ask questions,
or suggest additional steps.
"""


In [8]:
planner_agent = create_agent(
    model=model,
    tools=[
        search_models,
        verify_model_ids,
        save_evaluation_plan
    ],
    system_prompt=PLANNER_PROMPT,
)

In [9]:
# Planner Agent - Test

query = """
I am building an AI customer support chatbot for an e-commerce website.
The chatbot should provide clear and accurate answers,
follow instructions well,
respond quickly,
and keep API costs low.
I want to compare multiple LLMs and choose the best one for production.
"""

result = planner_agent.invoke({
    "messages": [HumanMessage(content=query)]
})

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================


I am building an AI customer support chatbot for an e-commerce website.
The chatbot should provide clear and accurate answers,
follow instructions well,
respond quickly,
and keep API costs low.
I want to compare multiple LLMs and choose the best one for production.

================================== Ai Message ==================================
Tool Calls:
  search_models (call_1544473)
 Call ID: call_1544473
  Args:
    query: fast low cost customer support models OpenRouter
================================= Tool Message =================================
Name: search_models

{"query": "fast low cost customer support models OpenRouter", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.youtube.com/watch?v=7dEkKSaQiXw", "title": "How To Create Production Customer Support AI agents with OpenRouter", "content": "With OpenRouter, we have multiple models available an

# Agent 2 - Benchmark Agent

In [10]:

# Benchmark Agent - Tools

@tool
def load_evaluation_plan() -> str:
    """Load and validate the evaluation plan created by the Planner Agent."""

    with open("evaluation_plan.json", "r", encoding="utf-8") as file:
        plan = json.load(file)

    validated_plan = EvaluationPlan.model_validate(plan)

    return json.dumps(
        validated_plan.model_dump(),
        ensure_ascii=False
    )


@tool(args_schema=BenchmarkDocument)
def save_benchmark(
    project_name: str,
    test_cases: list[dict]
) -> str:
    """Validate and save benchmark test cases as a JSON file."""

    benchmark = BenchmarkDocument.model_validate(
        {
            "project_name": project_name,
            "test_cases": test_cases
        }
    )

    with open("benchmark.json", "w", encoding="utf-8") as file:
        json.dump(
            benchmark.model_dump(),
            file,
            indent=2,
            ensure_ascii=False
        )

    return "Benchmark saved to benchmark.json"


In [11]:

# Benchmark Agent - Prompt

BENCHMARK_PROMPT = """
You are the Benchmark Agent in ModelRank.

Your task is to create a compact, project-specific benchmark.

First, call load_evaluation_plan to read the evaluation plan.

Generate test cases based only on the project's goal
and evaluation criteria.

Create tests only for criteria that require model responses,
such as accuracy, reasoning, clarity, or instruction following.

Do not create tests for latency or API cost.
These are measured during execution.

Make every test case self-contained and unambiguous.

Include all necessary facts, rules, policies, and context
inside the test prompt whenever factual accuracy is evaluated.

Do not invent external facts, policies, prices, dates,
or requirements missing from the evaluation plan.

Each expected_behavior must:
- Be specific and objectively verifiable.
- Follow only the facts and instructions in its test prompt.
- Avoid assumptions about missing information.
- Clearly identify when the model must not invent details.

Ensure every expected_behavior is logically consistent
with its corresponding prompt.

Create exactly 3 distinct, relevant test cases.
Avoid duplicates and unnecessary complexity.

Do not modify candidate models, run evaluations,
or select the best model.

Call save_benchmark to validate and save the test cases
as benchmark.json.

Do not output or summarize the test cases.

After saving, respond with exactly:
"Benchmark saved to benchmark.json"

Do not ask questions or suggest additional steps.
"""


In [12]:
# Benchmark Agent - Create Agent

benchmark_agent = create_agent(
    model=model,
    tools=[load_evaluation_plan, save_benchmark],
    system_prompt=BENCHMARK_PROMPT,
)

In [13]:
# Benchmark Agent - Test

result = benchmark_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Create the benchmark using the saved evaluation plan."
            )
        ]
    }
)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Create the benchmark using the saved evaluation plan.
================================== Ai Message ==================================
Tool Calls:
  load_evaluation_plan (call_3433437)
 Call ID: call_3433437
  Args:
================================= Tool Message =================================
Name: load_evaluation_plan

{"project_name": "E-commerce Customer Support Chatbot", "project_type": "Customer Support Chatbot", "project_goal": "Compare multiple LLMs and choose the best one for production customer support on an e-commerce website.", "evaluation_criteria": ["Clear and accurate answers", "Instruction following", "Response speed", "API cost"], "candidate_models": [{"name": "GPT-4o-mini", "provider": "OpenAI", "model_id": "openai/gpt-4o-mini"}, {"name": "Llama 3.3 70B Instruct", "provider": "Meta", "model_id": "meta-llama/llama-3.3-70b-instruct"}, {"name": "Claude Haiku 4.5", "provider": "Anthropic",

# Agent 3 - Runner Agent

In [14]:

# Runner Agent - Tools


@tool
def load_runner_inputs() -> str:
    """Load and validate the evaluation plan and benchmark."""

    with open("evaluation_plan.json", "r", encoding="utf-8") as file:
        plan = json.load(file)

    with open("benchmark.json", "r", encoding="utf-8") as file:
        benchmark = json.load(file)

    plan = EvaluationPlan.model_validate(plan).model_dump()
    benchmark = BenchmarkDocument.model_validate(benchmark).model_dump()

    inputs = {
        "evaluation_plan": plan,
        "benchmark": benchmark
    }

    return json.dumps(inputs, ensure_ascii=False)


@tool
def run_models_parallel(runner_inputs: str) -> str:
    """Run all candidate models on the same benchmark in parallel."""

    inputs = json.loads(runner_inputs)

    plan = EvaluationPlan.model_validate(
        inputs["evaluation_plan"]
    ).model_dump()

    benchmark = BenchmarkDocument.model_validate(
        inputs["benchmark"]
    ).model_dump()

    candidate_models = plan["candidate_models"]
    test_cases = benchmark["test_cases"]

    api_key = os.environ.get("OPENROUTER_API_KEY")
    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    def run_model(model_info):
        model_results = []

        for test_case in test_cases:

            payload = {
                "model": model_info["model_id"],
                "messages": [
                    {
                        "role": "user",
                        "content": test_case["prompt"]
                    }
                ],
                "temperature": 0,
                "usage": {
                    "include": True
                }
            }

            start_time = time.perf_counter()

            try:
                response = requests.post(
                    url,
                    headers=headers,
                    json=payload,
                    timeout=120
                )

                response.raise_for_status()

                elapsed_time = time.perf_counter() - start_time
                data = response.json()

                output = data["choices"][0]["message"]["content"]
                usage = data.get("usage", {})

                model_results.append(
                    {
                        "test_case_id": test_case["id"],
                        "criterion": test_case["criterion"],
                        "expected_behavior": test_case["expected_behavior"],
                        "output": output,
                        "latency_seconds": round(elapsed_time, 3),
                        "input_tokens": usage.get("prompt_tokens"),
                        "output_tokens": usage.get("completion_tokens"),
                        "total_tokens": usage.get("total_tokens"),
                        "cost_usd": usage.get("cost"),
                        "error": None
                    }
                )

            except Exception as error:
                elapsed_time = time.perf_counter() - start_time

                model_results.append(
                    {
                        "test_case_id": test_case["id"],
                        "criterion": test_case["criterion"],
                        "expected_behavior": test_case["expected_behavior"],
                        "output": None,
                        "latency_seconds": round(elapsed_time, 3),
                        "input_tokens": None,
                        "output_tokens": None,
                        "total_tokens": None,
                        "cost_usd": None,
                        "error": str(error)
                    }
                )

        return {
            "name": model_info["name"],
            "provider": model_info["provider"],
            "model_id": model_info["model_id"],
            "results": model_results
        }

    all_results = []

    with ThreadPoolExecutor(
        max_workers=len(candidate_models)
    ) as executor:

        futures = [
            executor.submit(
                run_model,
                model_info
            )
            for model_info in candidate_models
        ]

        for future in as_completed(futures):
            all_results.append(
                future.result()
            )

    results = {
        "project_name": plan["project_name"],
        "models": all_results
    }

    validated_results = RunnerResults.model_validate(results)

    return validated_results.model_dump_json()


@tool
def save_runner_results(results_json: str) -> str:
    """Calculate model summaries and save validated runner results."""

    runner_results = RunnerResults.model_validate_json(
        results_json
    ).model_dump()

    for model in runner_results["models"]:

        successful_results = [
            result
            for result in model["results"]
            if result["error"] is None
        ]

        failed_results = [
            result
            for result in model["results"]
            if result["error"] is not None
        ]

        latencies = [
            result["latency_seconds"]
            for result in successful_results
        ]

        costs = [
            result["cost_usd"]
            for result in successful_results
            if result["cost_usd"] is not None
        ]

        input_tokens = [
            result["input_tokens"]
            for result in successful_results
            if result["input_tokens"] is not None
        ]

        output_tokens = [
            result["output_tokens"]
            for result in successful_results
            if result["output_tokens"] is not None
        ]

        model["summary"] = {
            "average_latency_seconds": (
                round(sum(latencies) / len(latencies), 3)
                if latencies
                else None
            ),
            "total_cost_usd": (
                round(sum(costs), 8)
                if costs
                else None
            ),
            "total_input_tokens": sum(input_tokens),
            "total_output_tokens": sum(output_tokens),
            "successful_tests": len(successful_results),
            "failed_tests": len(failed_results)
        }

    with open("runner_results.json", "w", encoding="utf-8") as file:
        json.dump(
            runner_results,
            file,
            indent=2,
            ensure_ascii=False
        )

    return "Runner results saved to runner_results.json"


In [15]:

# Runner Agent - Prompt

RUNNER_PROMPT = """
You are the Runner Agent in ModelRank.

Your task is to execute the benchmark on all candidate models
and save the runtime results.

Always follow this tool order:

1. load_runner_inputs
2. run_models_parallel
3. save_runner_results

Pass the output of each tool directly to the next tool.

Use the same benchmark for every candidate model.
Models must be executed in parallel.

Do not modify candidate models, test cases,
expected behaviors, or evaluation criteria.

Collect model responses, latency, token usage,
API costs when available, and execution errors.

The save_runner_results tool calculates model summaries
and saves them to runner_results.json.

Do not evaluate response quality, assign PASS or FAIL,
rank models, or make recommendations.
Your responsibility is execution and data collection only.

After saving successfully, respond with exactly:
"Benchmark completed and saved to runner_results.json"

Do not print or summarize results.
Do not ask questions or suggest additional steps.
"""


In [16]:
# Runner Agent - Create Agent

runner_agent = create_agent(
    model=model,
    tools=[
        load_runner_inputs,
        run_models_parallel,
        save_runner_results
    ],
    system_prompt=RUNNER_PROMPT,
)

In [22]:
# Runner Agent - Test

result = runner_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Run the benchmark using the saved evaluation plan and benchmark."
            )
        ]
    }
)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Run the benchmark using the saved evaluation plan and benchmark.
================================== Ai Message ==================================
Tool Calls:
  load_runner_inputs (call_1002144)
 Call ID: call_1002144
  Args:
================================= Tool Message =================================
Name: load_runner_inputs

{"evaluation_plan": {"project_name": "E-commerce Customer Support Chatbot", "project_type": "Customer Support Chatbot", "project_goal": "Compare multiple LLMs and choose the best one for production customer support on an e-commerce website.", "evaluation_criteria": ["Clear and accurate answers", "Instruction following", "Response speed", "API cost"], "candidate_models": [{"name": "GPT-4o-mini", "provider": "OpenAI", "model_id": "openai/gpt-4o-mini"}, {"name": "Llama 3.3 70B Instruct", "provider": "Meta", "model_id": "meta-llama/llama-3.3-70b-instruct"}, {"name": "Claude Haiku 4.5

# Agent 4 - Judge Agent

In [23]:

# Judge Agent - Tools


@tool
def load_runner_results() -> str:
    """Load and validate the benchmark results produced by the Runner Agent."""

    with open("runner_results.json", "r", encoding="utf-8") as file:
        results = json.load(file)

    RunnerResults.model_validate(results)

    return json.dumps(
        results,
        ensure_ascii=False
    )


@tool(args_schema=FinalDecision)
def save_final_decision(
    selected_model: dict | None,
    model_rankings: list[dict],
    decision_reason: str,
    retry_required: bool,
    retry_reason: str = ""
) -> str:
    """Validate and save the Judge Agent's final model selection decision."""

    decision = FinalDecision.model_validate(
        {
            "selected_model": selected_model,
            "model_rankings": model_rankings,
            "decision_reason": decision_reason,
            "retry_required": retry_required,
            "retry_reason": retry_reason
        }
    )

    with open("final_decision.json", "w", encoding="utf-8") as file:
        json.dump(
            decision.model_dump(),
            file,
            indent=2,
            ensure_ascii=False
        )

    return "Final decision saved to final_decision.json"


In [24]:

# Judge Agent - Prompt

JUDGE_PROMPT = """
You are the Judge Agent in ModelRank.

Your task is to evaluate benchmark results, rank candidate models,
and recommend the best model for the user's project.

First, call load_runner_results.

Evaluate each model using:
- Compliance with expected_behavior
- Factual accuracy and instruction following
- Response latency and API cost
- Execution errors

Judge only from ModelRank benchmark evidence.
Do not rely on model or provider reputation.

Compare each response directly against its expected_behavior.
A successful API call does not mean the response passed the test.

Prioritize response correctness and instruction following.
When quality is similar, prefer lower latency and cost.

Rank all candidate models from best to worst.

Select the best-performing model if at least one candidate
meets the project's important requirements.

Set retry_required to true only when all candidates
perform poorly or fail important requirements.

If retry_required is true:
- Do not recommend a selected model.
- Explain why the candidates are insufficient.

If retry_required is false:
- Select one best-performing model.
- Explain the choice using benchmark evidence.

Keep decision_reason concise and evidence-based.
Avoid unsupported claims of perfect or flawless performance.

Do not modify results, rerun models, create test cases,
or search for new models.

Call save_final_decision to save final_decision.json.

After saving, respond with exactly:
"Final decision saved to final_decision.json"

Do not summarize results, ask questions,
or suggest additional steps.
"""


In [25]:
# Judge Agent - Create Agent

judge_agent = create_agent(
    model=model,
    tools=[
        load_runner_results,
        save_final_decision
    ],
    system_prompt=JUDGE_PROMPT,
)

In [26]:
# Judge Agent - Test

result = judge_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Evaluate the saved runner results and select the best model."
            )
        ]
    }
)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Evaluate the saved runner results and select the best model.
================================== Ai Message ==================================
Tool Calls:
  load_runner_results (call_4691152)
 Call ID: call_4691152
  Args:
================================= Tool Message =================================
Name: load_runner_results

{"project_name": "E-commerce Customer Support Chatbot", "models": [{"name": "Gemini 2.5 Flash", "provider": "Google", "model_id": "google/gemini-2.5-flash", "results": [{"test_case_id": "cs_accuracy_return_policy", "criterion": "Clear and accurate answers", "expected_behavior": "The response must state that the customer can return the headphones because 20 days is within the 30-day window. It must explicitly state that because the electronics item has been opened, it is subject to a 15% restocking fee (resulting in an 85% refund). The response must not invent any policy details, fe

# LangGraph

In [27]:
# LangGraph - State


class ModelRankState(TypedDict):
    user_request: str
    retry_count: int
    retry_required: bool
    selected_model: dict | None

In [28]:

# LangGraph - Nodes


def planner_node(state: ModelRankState):
    if state["retry_count"] > 0:
        planner_message = f"""
The previous candidate models were not good enough.

Select a new set of candidate models for the same user request.

Do not select the same candidate models used in the previous attempt.

Original user request:
{state["user_request"]}
"""
    else:
        planner_message = state["user_request"]

    # Remove previous plan to prevent stale results
    if os.path.exists("evaluation_plan.json"):
        os.remove("evaluation_plan.json")

    planner_agent.invoke(
        {
            "messages": [
                HumanMessage(content=planner_message)
            ]
        }
    )

    if not os.path.exists("evaluation_plan.json"):
        raise RuntimeError("Planner failed to save evaluation_plan.json")

    with open("evaluation_plan.json", "r", encoding="utf-8") as file:
        EvaluationPlan.model_validate(json.load(file))

    return state


def benchmark_node(state: ModelRankState):
    # Remove previous benchmark before generating a new one
    if os.path.exists("benchmark.json"):
        os.remove("benchmark.json")

    benchmark_agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content="Create the benchmark using the saved evaluation plan."
                )
            ]
        }
    )

    if not os.path.exists("benchmark.json"):
        raise RuntimeError("Benchmark failed to save benchmark.json")

    with open("benchmark.json", "r", encoding="utf-8") as file:
        BenchmarkDocument.model_validate(json.load(file))

    return state


def runner_node(state: ModelRankState):
    # Remove previous results before execution
    if os.path.exists("runner_results.json"):
        os.remove("runner_results.json")

    runner_agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content="Run the benchmark using the saved evaluation plan and benchmark."
                )
            ]
        }
    )

    if not os.path.exists("runner_results.json"):
        raise RuntimeError("Runner failed to save runner_results.json")

    with open("runner_results.json", "r", encoding="utf-8") as file:
        results = json.load(file)

    RunnerResults.model_validate(results)

    if any(model.get("summary") is None for model in results["models"]):
        raise RuntimeError("Runner results are missing model summaries")

    return state


def judge_node(state: ModelRankState):
    # Remove previous decision before evaluation
    if os.path.exists("final_decision.json"):
        os.remove("final_decision.json")

    judge_agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content="Evaluate the saved runner results and select the best model."
                )
            ]
        }
    )

    if not os.path.exists("final_decision.json"):
        raise RuntimeError("Judge failed to save final_decision.json")

    with open("final_decision.json", "r", encoding="utf-8") as file:
        decision = FinalDecision.model_validate(
            json.load(file)
        ).model_dump()

    retry_count = state["retry_count"]

    if decision["retry_required"]:
        retry_count += 1

    return {
        **state,
        "retry_required": decision["retry_required"],
        "retry_count": retry_count,
        "selected_model": decision["selected_model"]
    }


In [29]:
# LangGraph - Routing


MAX_RETRIES = 1


def route_after_planner(state: ModelRankState):
    if state["retry_count"] > 0:
        return "runner"

    return "benchmark"


def route_after_judge(state: ModelRankState):
    if (
        state["retry_required"]
        and state["retry_count"] < MAX_RETRIES
    ):
        return "planner"

    return "end"

In [30]:
# LangGraph - Workflow


workflow = StateGraph(ModelRankState)

workflow.add_node("planner", planner_node)
workflow.add_node("benchmark", benchmark_node)
workflow.add_node("runner", runner_node)
workflow.add_node("judge", judge_node)

workflow.set_entry_point("planner")

workflow.add_conditional_edges(
    "planner",
    route_after_planner,
    {
        "benchmark": "benchmark",
        "runner": "runner"
    }
)

workflow.add_edge(
    "benchmark",
    "runner"
)

workflow.add_edge(
    "runner",
    "judge"
)

workflow.add_conditional_edges(
    "judge",
    route_after_judge,
    {
        "planner": "planner",
        "end": END
    }
)

model_fit_graph = workflow.compile()

In [31]:
# LangGraph - End-to-End Test


initial_state = {
    "user_request": """
I am building an AI customer support chatbot for an e-commerce website.

The chatbot should provide clear and accurate answers,
follow instructions well,
respond quickly,
and keep API costs low.

I want to compare multiple LLMs and choose the best one for production.
""",
    "retry_count": 0,
    "retry_required": False,
    "selected_model": None
}

final_state = model_fit_graph.invoke(initial_state)

print(final_state)

{'user_request': '\nI am building an AI customer support chatbot for an e-commerce website.\n\nThe chatbot should provide clear and accurate answers,\nfollow instructions well,\nrespond quickly,\nand keep API costs low.\n\nI want to compare multiple LLMs and choose the best one for production.\n', 'retry_count': 0, 'retry_required': False, 'selected_model': {'name': 'Llama 3.1 8B Instruct', 'provider': 'Meta', 'model_id': 'meta-llama/llama-3.1-8b-instruct'}}


In [32]:
# LangGraph - Retry Routing Test


test_state = {
    "user_request": "Test request",
    "retry_count": 1,
    "retry_required": True,
    "selected_model": None
}

print(route_after_judge(test_state))

end


In [33]:
# LangGraph - Retry Limit Test


test_state = {
    "user_request": "Test request",
    "retry_count": 2,
    "retry_required": True,
    "selected_model": None
}

print(route_after_judge(test_state))

end
